## Image Segmentation
Converting the pdf to image for detecting the images in the page

In [1]:
from pdf2image import convert_from_path
import os

INPUT_PDF = "../input/sample_1.pdf"
OUTPUT_DIR = "../output/page_images"

os.makedirs(OUTPUT_DIR, exist_ok=True)


pages = convert_from_path(INPUT_PDF, dpi=300)  # high-res for figure clarity

for i, page in enumerate(pages):
    page_path = f"{OUTPUT_DIR}/page_{i+1}.png"
    page.save(page_path, "PNG")
    print(f"Saved {page_path}")


Saved ../output/page_images/page_1.png


In [2]:
import cv2
import numpy as np
import os

PAGE_IMAGES_DIR = "../output/page_images"
FIGURES_DIR = "../output/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

for img_file in os.listdir(PAGE_IMAGES_DIR):
    img_path = os.path.join(PAGE_IMAGES_DIR, img_file)
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        # Skip very small areas (probably text)
        if w < 50 or h < 50:
            continue
        figure_crop = img[y:y+h, x:x+w]
        out_path = os.path.join(FIGURES_DIR, f"{img_file[:-4]}_fig{count+1}.png")
        cv2.imwrite(out_path, figure_crop)
        count += 1
    print(f"Detected {count} figures in {img_file}")


Detected 3 figures in page_1.png


## Image to Text

In [3]:
from groq import Groq
import base64
import os

# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

# Path to your image
image_path = "../output/figures/page_1_fig3.png"

# Getting the base64 string
base64_image = encode_image(image_path)

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "This is the image within a research paper page. You need to give me a very short description of what this image is about. If possible finish in one line."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                    },
                },
            ],
        }
    ],
    model="meta-llama/llama-4-scout-17b-16e-instruct",
)

print(chat_completion.choices[0].message.content)

The image shows a graph with multiple lines representing different values of n, illustrating how a variable changes with $\tau/n$.


## Formula Detection

In [4]:
import fitz  # PyMuPDF
import json
import os

INPUT_PDF = "../input/sample_1.pdf"
OUTPUT_DIR = "../output/formulas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

doc = fitz.open(INPUT_PDF)
formulas = []

# Math symbols to detect
MATH_SYMBOLS = ["=", "∑", "∫", "λ", "→", "^", "_", "|", "α", "β", "γ", "+", "-", "*", "/"]

# Heuristic function
def is_formula(block, page_height):
    text = " ".join(span["text"] for line in block["lines"] for span in line["spans"]).strip()
    if not text:
        return False
    
    # Symbol density
    if not any(sym in text for sym in MATH_SYMBOLS):
        return False
    
    # Shorter lines more likely formulas
    if len(text.split()) > 25:
        return False
    
    # Check vertical position: centered formulas often around middle of page
    y0, y1 = block["bbox"][1], block["bbox"][3]  # top, bottom of block
    block_center = (y0 + y1) / 2
    if not (0.2*page_height <= block_center <= 0.8*page_height):
        return False
    
    return True

# Extract formulas
for page_num, page in enumerate(doc):
    blocks = page.get_text("dict")["blocks"]
    page_height = page.rect.height

    for block in blocks:
        if block["type"] == 0:  # text block
            if is_formula(block, page_height):
                text = " ".join(span["text"] for line in block["lines"] for span in line["spans"]).strip()
                formulas.append({
                    "page": page_num + 1,
                    "formula": text,
                    "bbox": block["bbox"]
                })

# Save to JSON
with open(f"{OUTPUT_DIR}/formulas.json", "w") as f:
    json.dump(formulas, f, indent=2)

print(f"Detected {len(formulas)} formulas.")


Detected 7 formulas.


In [5]:
formulas[0].get("formula")


'n  = 256 n  = 512 n  = 1024 n  = 2048 n  = 4096'

In [6]:
formulas


[{'page': 1,
  'formula': 'n  = 256 n  = 512 n  = 1024 n  = 2048 n  = 4096',
  'bbox': (171.73468017578125,
   163.5403594970703,
   196.83123779296875,
   208.421875)},
 {'page': 1,
  'formula': 'n  = 256 n  = 512 n  = 1024 n  = 2048 n  = 4096',
  'bbox': (171.73468017578125,
   163.5403594970703,
   196.83123779296875,
   208.421875)},
 {'page': 1,
  'formula': 'W  = 64 W  = 128 W  = 256',
  'bbox': (382.7744445800781,
   177.6538543701172,
   406.8191833496094,
   203.4605255126953)},
 {'page': 1,
  'formula': 'W  = 64 W  = 128 W  = 256',
  'bbox': (382.7744445800781,
   177.6538543701172,
   406.8191833496094,
   203.4605255126953)},
 {'page': 1,
  'formula': 'I ( t ) =   β t',
  'bbox': (216.552001953125,
   338.8079528808594,
   258.65069580078125,
   355.5105285644531)},
 {'page': 1,
  'formula': 'µ =1',
  'bbox': (264.4670104980469,
   359.51776123046875,
   279.4049072265625,
   366.4915771484375)},
 {'page': 1,
  'formula': '\x02 ˜ x µ s θ (˜ x µ , t ) +  s θ (˜ x µ , t ) 2 \